In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTENC

In [2]:
paths = [
    Path("data/raw/student_mental_health.csv"),
    Path("../data/raw/student_mental_health.csv"),
    Path("student_mental_health.csv")
]

csv_path = next((p for p in paths if p.exists()), None)
if csv_path is None:
    raise FileNotFoundError("Place student_mental_health.csv in data/raw/")

df = pd.read_csv(csv_path)
print("Initial dimensions:", df.shape)
display(df.head(3))

Initial dimensions: (2000, 17)


,Student_ID,Age,Gender,Year_of_Study,Sleep_Hours,Study_Hours_Per_Week,GPA,Social_Media_Hours,Screen_Time_Hours,Physical_Activity,Financial_Stress,Family_Support,Attendance_Rate,Part_Time_Job,Diet_Quality,Previous_Mental_Health_History,Mental_Health_Risk
0,STU0001,22,Male,3,6.6,17.6,2.91,6.9,7.5,Moderate,8.5,4.0,78.0,No,Poor,No,High
1,STU0002,21,Male,4,5.6,22.4,2.92,7.6,11.5,Low,4.5,5.3,85.4,Yes,Average,No,Medium
2,STU0003,22,Male,5,7.1,28.1,2.68,6.5,8.9,Low,4.8,6.1,79.5,No,Average,No,Medium


In [3]:
target = "Mental_Health_Risk"

print("Missing target values:", df[target].isna().sum())

df = df.dropna(subset=[target]).copy()
X = df.drop(columns=["Student_ID", target])
y = df[target]

print("Input features:", X.shape[1])
print("Classes:", sorted(y.unique()))

Missing target values: 0
Input features: 15
Classes: ['High', 'Low', 'Medium']


In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("X_train :", X_train.shape)
print("X_test  :", X_test.shape)
print("\nTraining class distribution:")
print(y_train.value_counts())
print("\nTest class distribution:")
print(y_test.value_counts())

X_train : (1600, 15)
X_test  : (400, 15)

Training class distribution:
Mental_Health_Risk
High      544
Medium    528
Low       528
Name: count, dtype: int64

Test class distribution:
Mental_Health_Risk
High      136
Medium    132
Low       132
Name: count, dtype: int64


In [5]:
numeric_columns = X_train.select_dtypes(include="number").columns.tolist()
categorical_columns = X_train.select_dtypes(exclude="number").columns.tolist()

print("Numerical columns:", numeric_columns)
print("\nCategorical columns:", categorical_columns)

Numerical columns: ['Age', 'Year_of_Study', 'Sleep_Hours', 'Study_Hours_Per_Week', 'GPA', 'Social_Media_Hours', 'Screen_Time_Hours', 'Financial_Stress', 'Family_Support', 'Attendance_Rate']

Categorical columns: ['Gender', 'Physical_Activity', 'Part_Time_Job', 'Diet_Quality', 'Previous_Mental_Health_History']


In [6]:
print("Missing values in X_train:")
print(X_train.isna().sum().sort_values(ascending=False))

print("\nMissing values in X_test:")
print(X_test.isna().sum().sort_values(ascending=False))

Missing values in X_train:
Financial_Stress                  76
Diet_Quality                      55
Sleep_Hours                       54
Social_Media_Hours                48
Attendance_Rate                   39
Family_Support                    36
GPA                               36
Study_Hours_Per_Week               0
Age                                0
Gender                             0
Year_of_Study                      0
Screen_Time_Hours                  0
Physical_Activity                  0
Part_Time_Job                      0
Previous_Mental_Health_History     0
dtype: int64

Missing values in X_test:
Family_Support                    18
Financial_Stress                  15
Diet_Quality                      14
Social_Media_Hours                10
Attendance_Rate                    8
Sleep_Hours                        8
GPA                                2
Study_Hours_Per_Week               0
Age                                0
Gender                             0
Year_of_

In [7]:
numeric_imputer = SimpleImputer(strategy="median")
categorical_imputer = SimpleImputer(strategy="most_frequent")

print("Numerical imputation: median")
print("Categorical imputation: most frequent value")

Numerical imputation: median
Categorical imputation: most frequent value


In [8]:
q1 = X_train[numeric_columns].quantile(0.25)
q3 = X_train[numeric_columns].quantile(0.75)
iqr = q3 - q1

lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

outliers = (
    (X_train[numeric_columns] < lower_bound) |
    (X_train[numeric_columns] > upper_bound)
).sum()

print(outliers.sort_values(ascending=False))

Attendance_Rate         10
Social_Media_Hours      10
Sleep_Hours              9
Study_Hours_Per_Week     7
Screen_Time_Hours        7
GPA                      5
Age                      2
Year_of_Study            0
Financial_Stress         0
Family_Support           0
dtype: int64


In [9]:
class IQRClipper(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        X = pd.DataFrame(X)
        q1 = X.quantile(0.25)
        q3 = X.quantile(0.75)
        iqr = q3 - q1
        self.lower_bounds_ = q1 - 1.5 * iqr
        self.upper_bounds_ = q3 + 1.5 * iqr
        return self

    def transform(self, X):
        X = pd.DataFrame(X)
        return X.clip(self.lower_bounds_, self.upper_bounds_, axis=1).to_numpy()

In [10]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("iqr", IQRClipper()),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

In [11]:
preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_columns),
    ("cat", categorical_pipeline, categorical_columns)
])

print(preprocessor)

ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('iqr', IQRClipper()),
                                                 ('scaler', StandardScaler())]),
                                 ['Age', 'Year_of_Study', 'Sleep_Hours',
                                  'Study_Hours_Per_Week', 'GPA',
                                  'Social_Media_Hours', 'Screen_Time_Hours',
                                  'Financial_Stress', 'Family_Support',
                                  'Attendance_Rate']),
                                ('cat',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('encoder',
                                                  OneHotEncoder(h

In [12]:
X_train_prepared = preprocessor.fit_transform(X_train)
X_test_prepared = preprocessor.transform(X_test)

print("Prepared training set:", X_train_prepared.shape)
print("Prepared test set:", X_test_prepared.shape)

print("Missing values after training preprocessing:", np.isnan(X_train_prepared).sum())
print("Missing values after test preprocessing:", np.isnan(X_test_prepared).sum())

print("Numeric output after preprocessing:", np.issubdtype(X_train_prepared.dtype, np.number))

Prepared training set: (1600, 23)
Prepared test set: (400, 23)
Missing values after training preprocessing: 0
Missing values after test preprocessing: 0
Numeric output after preprocessing: True


In [13]:
print("Before SMOTE:")
print(y_train.value_counts())

print("\nClass proportions (%):")
print((y_train.value_counts(normalize=True) * 100).round(2))

Before SMOTE:
Mental_Health_Risk
High      544
Medium    528
Low       528
Name: count, dtype: int64

Class proportions (%):
Mental_Health_Risk
High      34.0
Medium    33.0
Low       33.0
Name: proportion, dtype: float64


In [14]:
X_demo_smote = X_train.copy()

X_demo_smote[numeric_columns] = preprocessor.named_transformers_["num"].transform(
    X_train[numeric_columns]
)

X_demo_smote[categorical_columns] = preprocessor.named_transformers_["cat"].named_steps[
    "imputer"
].transform(X_train[categorical_columns])

smote = SMOTENC(categorical_features=categorical_columns, random_state=42)
X_smote, y_smote = smote.fit_resample(X_demo_smote, y_train)

print("Before:", y_train.value_counts().to_dict())
print("After:", pd.Series(y_smote).value_counts().to_dict())
print("Number of rows before:", len(X_train))
print("Number of rows after:", len(X_smote))

Before:

 {'High': 544, 'Medium': 528, 'Low': 528}
After: {'Medium': 544, 'Low': 544, 'High': 544}
Number of rows before: 1600
Number of rows after: 1632
